# 03 — Modeling: cost-aware selection across families

Display the training-pipeline outcome: per-family Optuna results, the cost-weighted Precision@k winner, calibration curves, and the cost-vs-threshold sweep. Reads from `mlruns/training_summary.json` produced by `scripts/train.sh`.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', palette='deep')
summary_path = Path('mlruns/training_summary.json')
summary = json.loads(summary_path.read_text())
summary['winning_family']

## 1. Per-family comparison

The cost-weighted Precision@k objective at the best Optuna trial for each family. The selection ranks families on the *operational* objective, not AUC-PR — see `docs/EVALUATION.md`.

In [ ]:
families = summary['all_families']
comparison = pd.DataFrame([
    {'family': name, 'objective_usd_per_hour': -float(info['objective'])}
    for name, info in families.items()
]).sort_values('objective_usd_per_hour')

fig, ax = plt.subplots(figsize=(9, 4))
colours = ['#22c55e' if name == summary['winning_family'] else '#94a3b8' for name in comparison['family']]
ax.barh(comparison['family'], comparison['objective_usd_per_hour'], color=colours)
ax.set_xlabel('Cost per investigator-hour (USD, lower is better)')
ax.set_title('Per-family cost-weighted objective (validation fold)')
plt.tight_layout()
plt.show()
comparison

## 2. Test-fold headline metrics

In [ ]:
test_eval = summary['test']
tp = int(test_eval['true_positives'])
fp = int(test_eval['false_positives'])
fn = int(test_eval['false_negatives'])
recall = tp / max(tp + fn, 1)

print(f'Decision threshold: {summary["decision_threshold"]:.4f}')
print(f'k (alerts reviewed):    {int(test_eval["k"]):,}')
print(f'True positives:         {tp:,} ({recall * 100:.2f}% recall)')
print(f'False positives:        {fp:,}')
print(f'False negatives:        {fn:,}')
print(f'Precision @ k:          {test_eval["precision_at_k"] * 100:.2f}%')
print(f'Total dollar cost:      ${test_eval["total_dollar_cost_usd"]:,.2f}')

## 3. Cost matrix transparency

The cost matrix the selection optimised against. Every value here is documented in `configs/cost_matrix.yaml` with its derivation.

In [ ]:
cm = summary['cost_matrix']
print(f'Daily review capacity (k):     {cm["k_per_day"]:,} alerts')
print(f'False-negative cost (per miss): ${cm["false_negative_cost_usd"]:,.2f}')
print(f'False-positive cost (per FP):   ${cm["false_positive_cost_usd"]:.2f}')

## 4. Winning model: hyperparameter snapshot

In [ ]:
import pprint
print(f'Selected family: {summary["winning_family"]}')
pprint.pprint(summary['winning_hyperparameters'])